# Golden JSON intersection

Intersect the run/lumi coverage extracted from the skimmed data files with the official CMS 2018 Golden JSON.


In [7]:
import json
import os


input_json_path = (
    "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/"
    "processed_lumis_DoubleMuon_2018ABCD_merged.json"
)

golden_json_path = (
    "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/"
    "Cert_314472-325175_13TeV_Legacy2018_Collisions18_JSON.txt"
)

output_json_path = (
    "/home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/"
    "processed_lumis_DoubleMuon_2018ABCD_merged_golden.json"
)


In [8]:
def expand_ranges(ranges):
    lumis = set()
    for start, end in ranges:
        lumis.update(range(int(start), int(end) + 1))
    return lumis


def compress_lumis(lumis):
    lumis = sorted(lumis)
    if not lumis:
        return []

    ranges = []
    start = prev = lumis[0]

    for lumi in lumis[1:]:
        if lumi == prev + 1:
            prev = lumi
        else:
            ranges.append([start, prev])
            start = prev = lumi

    ranges.append([start, prev])
    return ranges


def load_json_as_sets(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"JSON file not found: {path}")

    with open(path) as f:
        data = json.load(f)

    run_lumis = {}
    for run, ranges in data.items():
        try:
            run_number = str(int(run))
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Invalid run key in {path}: {run!r}") from exc
        run_lumis[run_number] = expand_ranges(ranges)

    return run_lumis


def count_lumisections(run_lumis):
    return sum(len(lumis) for lumis in run_lumis.values())


In [9]:
skim_lumis = load_json_as_sets(input_json_path)
golden_lumis = load_json_as_sets(golden_json_path)

intersected_lumis = {}
for run in sorted(set(skim_lumis) & set(golden_lumis), key=int):
    good_lumis = skim_lumis[run] & golden_lumis[run]
    if good_lumis:
        intersected_lumis[run] = good_lumis

output_json = {
    run: compress_lumis(lumis)
    for run, lumis in intersected_lumis.items()
}

os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
with open(output_json_path, "w") as f:
    json.dump(output_json, f, indent=2)

skim_count = count_lumisections(skim_lumis)
golden_count = count_lumisections(golden_lumis)
output_count = count_lumisections(intersected_lumis)

print(f"Input JSON       : {input_json_path}")
print(f"Golden JSON      : {golden_json_path}")
print(f"Output JSON      : {output_json_path}")
print()
print(f"Input runs       : {len(skim_lumis)}")
print(f"Golden runs      : {len(golden_lumis)}")
print(f"Intersected runs : {len(intersected_lumis)}")
print(f"Input lumis      : {skim_count}")
print(f"Golden lumis     : {golden_count}")
print(f"Intersected lumis: {output_count}")
print(f"Removed lumis    : {skim_count - output_count}")


Input JSON       : /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/processed_lumis_DoubleMuon_2018ABCD_merged.json
Golden JSON      : /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/Cert_314472-325175_13TeV_Legacy2018_Collisions18_JSON.txt
Output JSON      : /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/JSON/Merged/processed_lumis_DoubleMuon_2018ABCD_merged_golden.json

Input runs       : 542
Golden runs      : 478
Intersected runs : 478
Input lumis      : 253739
Golden lumis     : 234527
Intersected lumis: 232840
Removed lumis    : 20899
